### SHAP Explainability

SHAP was used to explain the model's prediction by identifying the most influential features for the predicted attack class.

This improves model interpretability and shows **why the Random Forest made its decision**.


In [27]:
import numpy as np
import shap
import pandas as pd

In [12]:
import joblib

model = joblib.load("models/rf_model.pkl")
scaler = joblib.load("models/scaler.pkl")
label_encoder = joblib.load("models/label_encoder.pkl")
feature_names = joblib.load("models/feature_names.pkl")
X_test_z = joblib.load("X_test_z.pkl")

In [13]:
y_pred = model.predict(X_test_z)


In [16]:
attack = label_encoder.inverse_transform(y_pred)[0]

confidence = np.max(model.predict_proba(X_test_z)) * 100

print(f"Attack: {attack}")
print(f"Confidence: {confidence:.2f}%")

Attack: DoS
Confidence: 100.00%


In [19]:
idx = 2

row = X_test_z.iloc[[idx]]

pred_encoded = model.predict(row)[0]
pred_proba = model.predict_proba(row)[0]

attack = label_encoder.inverse_transform([pred_encoded])[0]
confidence = np.max(pred_proba) * 100

print(f"Attack: {attack}")
print(f"Confidence: {confidence:.2f}%")

Attack: PortScan
Confidence: 99.72%


In [20]:
type(X_test_z)
X_test_z.shape

(374729, 17)

# SHAP XAI

In [23]:
# explainer 
explainer = shap.TreeExplainer(model)

In [24]:
sample = X_test_z.iloc[[0]]
shap_values = explainer.shap_values(sample)

In [25]:
if isinstance(shap_values, list):
    print("Number of classes:", len(shap_values))
else:
    print("Shape:", shap_values.shape)

Shape: (1, 17, 8)


In [26]:
pred_idx = model.predict(sample)[0]
pred_label = label_encoder.inverse_transform([pred_idx])[0]

print(f"Predicted class index: {pred_idx}")
print(f"Predicted label: {pred_label}")


Predicted class index: 4
Predicted label: DoS


In [29]:
class_shap_values = shap_values[0, :, pred_idx]

impact_df = pd.DataFrame({
    "Feature": sample.columns,
    "Value": sample.iloc[0].values,
    "SHAP": class_shap_values
})

impact_df["Contribution"] = np.where(
    impact_df["SHAP"] > 0,
    "Positive",
    "Negative"
)

top_features = (
    impact_df.assign(Abs_SHAP=np.abs(impact_df["SHAP"]))
             .sort_values("Abs_SHAP", ascending=False)
             .drop(columns="Abs_SHAP")
             .head(5)
             .reset_index(drop=True)
)

top_features.insert(0, "Rank", range(1, len(top_features) + 1))

print(top_features)

   Rank                     Feature     Value      SHAP Contribution
0     1   Bwd Packet Length Std_log  1.879159  0.258676     Positive
1     2  Bwd Packet Length Mean_log  1.491164  0.131569     Positive
2     3       Max Packet Length_log  1.544389  0.089916     Positive
3     4               Idle Mean_log  2.018145  0.055976     Positive
4     5     Average Packet Size_log  1.403787  0.055958     Positive
